[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week04.ipynb)

# Week 4: Data Pipelines
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

The Dataset and DataLoader abstractions; batching, shuffling, transforms, and splits.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Build a custom Dataset and DataLoader.
- Reason about batching, shuffling, and clean splits.
- Recognize and avoid data leakage.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. A custom Dataset
Implement __len__ and __getitem__ to return one (input, label).

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ToyData(Dataset):
    def __init__(self, n=120):
        self.x = torch.randn(n, 3)
        self.y = (self.x.sum(1) > 0).long()
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

ds = ToyData()
print("len:", len(ds), "| one item:", ds[0][0].shape, ds[0][1].item())

## 2. TensorDataset is the quick version
When the data already fits in tensors, skip the class.

In [ ]:
from torch.utils.data import TensorDataset
X = torch.randn(120, 3); y = (X.sum(1) > 0).long()
quick = TensorDataset(X, y)
print("same interface:", len(quick), quick[0][0].shape, quick[0][1].item())

## 3. A DataLoader batches and shuffles
Iterating yields batches shaped (batch, ...).

In [ ]:
dl = DataLoader(ds, batch_size=16, shuffle=True)
xb, yb = next(iter(dl))
print("batch:", tuple(xb.shape), tuple(yb.shape))
for i, (xb, yb) in enumerate(dl):
    print("batch", i, "->", xb.shape[0], "samples")
    if i == 3:
        break

## 4. Batch size and shuffling
Batch size sets the steps per epoch; shuffling reorders every epoch.

In [ ]:
for bs in [8, 16, 64]:
    print(f"batch_size={bs}: {len(DataLoader(ds, batch_size=bs))} batches per epoch")
dl = DataLoader(ds, batch_size=4, shuffle=True)
print("epoch 1 first labels:", next(iter(dl))[1].tolist())
print("epoch 2 first labels:", next(iter(dl))[1].tolist())

## 5. Transforms preprocess each sample
A transform pipeline runs as each item is read.

In [ ]:
from torchvision import transforms
pipe = transforms.Compose([
    transforms.ToTensor(),                                  # PIL/np -> CxHxW float in [0,1]
    transforms.Normalize(mean=[0.5], std=[0.5]),            # -> roughly [-1, 1]
])
import numpy as np
fake = (np.random.rand(8, 8) * 255).astype("uint8")
out = pipe(fake)
print("after ToTensor + Normalize:", tuple(out.shape), "range", round(out.min().item(), 2), round(out.max().item(), 2))

## 6. A train / validation / test split
Tune on validation, touch test once. random_split is the easy way.

In [ ]:
from torch.utils.data import random_split
gen = torch.Generator().manual_seed(0)
tr, va, te = random_split(ds, [80, 20, 20], generator=gen)
print("sizes:", len(tr), len(va), len(te))
train_dl = DataLoader(tr, batch_size=16, shuffle=True)
val_dl   = DataLoader(va, batch_size=16)            # no shuffle for val/test

## 7. Imbalanced data: a weighted sampler
Oversample the rare class so each batch is roughly balanced.

In [ ]:
from torch.utils.data import WeightedRandomSampler
labels = torch.cat([torch.zeros(90), torch.ones(10)]).long()       # 90 vs 10
class_count = torch.bincount(labels).float()
weight_per_sample = (1.0 / class_count)[labels]                    # rare class -> higher weight
sampler = WeightedRandomSampler(weight_per_sample, num_samples=100, replacement=True)
drawn = labels[list(sampler)]
print("class counts in a sampled epoch:", torch.bincount(drawn).tolist(), "(was 90 / 10)")

## 8. Data leakage, demonstrated
Normalizing with whole-dataset statistics leaks test information.

In [ ]:
data = torch.randn(100, 1) * 5 + 10
train, test = data[:80], data[80:]

mu, sd = data.mean(), data.std()                 # LEAK: uses test data
print("leaky   test mean ~ 0:", round(((test - mu) / sd).mean().item(), 3))

mu, sd = train.mean(), train.std()               # correct: train only
print("correct test mean (not 0):", round(((test - mu) / sd).mean().item(), 3))

### Mini-exercise
Build a DataLoader over `ToyData` with batch_size 10, then compute the mean of every batch's labels. Are the batches roughly balanced when shuffle=True?

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
dl = DataLoader(ToyData(100), batch_size=10, shuffle=True)
means = [yb.float().mean().item() for _, yb in dl]
print("per-batch label means:", [round(m, 2) for m in means])
print("they hover near the dataset base rate, not 0 or 1")

**Recap:** fit preprocessing on the training split only; shuffle training, not val/test; batch size trades gradient noise against speed; leakage silently inflates results.

---
Student materials for this week: the lab handout (`labs/week04.html`) and the curated references (`references/week04.html`) in the course site.